# Kalman Filter Price-Target Model — Fused Panel with a Per-ISIN Latent (v4)

**Notebook form of `pymc_kalman_filter_pt.py`** (0.9.9.14), aligned with
`probabilistic_ml_model/pymc_models/KalmanFilterModel.py`. Supersedes
`pymc_kalman_filter_pt_v3.ipynb`, which tracked 0.9.9.11.

The cross-sectional spine is the **fused hierarchical panel model**
(`build_fused_kalman_pt_model`):

- **Model B spine** — a rank-1 Intrinsic Coregionalization Model (ICM) over the
  `(isin, time, y_series)` response tensor: the `D` response series share the latent
  per-ISIN factor `mu_isin` through per-series loadings `mu_isin_loading` (primary
  anchored at 1) with a per-series noise diagonal `sigma_series`. **The ICM is dormant
  while `D == 1`** (the default) — `KalmanRunConfig.panel_response_extra` is what
  activates it. Per-time levels are **T direct per-time intercepts** (`alpha_level`);
  the per-series time slope `beta_t` / `beta_slope` is materialised **only** when
  `t_scaled` genuinely varies across ISINs, which the `np.tile`-built T>1 axis does not.
- **Model A refinement** — the risk-aware `expected_return` latent is the structural
  mean, with the heteroscedastic scale `sigma_isin = sigma_base · (1 + cv) / √n` and the
  additive systematic-risk / size / volume tilts
  `risk_adj_return = ER − risk_loading·z(avg_beta) − size_loading·z(mcap_ratio) − volume_loading·z(rel_volume)`.
- **Per-ISIN random intercept** (`sigma_isin_level` × a non-centred `ZeroSumNormal`),
  restored for **`T > 1` only**. It was dropped in an earlier release as non-identified —
  correct *for the T=1 cross-section*; `T` repeated observations per name identify it in
  the ordinary way. `state_path` (isin, time) and `state_now = state_path[:, -1]` are
  always emitted, so the decision latent has exactly one name.

## What's new in v4 (CHANGELOG 0.9.9.13 / 0.9.9.14)

1. **`state_now` is the decision latent.** Every consumer — screen, price-target Monte
   Carlo, risk book, analytics export, §13b plots, prior predictive — resolves it through
   `KALMAN_SCREEN_LATENT` / `resolve_screen_latent`, with a documented fallback to
   `risk_adj_return` for pre-0.9.9.14 NetCDF artifacts. `achieve_prob = sigmoid(state_now)`.
2. **The genuine T=4 panel is now the DEFAULT** (`panel_lookbacks=('6m','3m','1m')`).
   v3 had this backwards: it was the opt-in there, and the collapsed T=1 snapshot is now
   the opt-*out*.
3. **Per-ISIN latent fixes a ~√T over-confidence.** Broadcasting a time-constant `mu_isin`
   across the T lookback slices treated each name's `T` serially-correlated observations as
   `T` iid draws. Measured on the live 5,605-name T=4 panel: `sigma_base` 0.3373 → 0.2871
   and mean per-name posterior sd 0.0545 → **0.2694**. On synthetic data with a known
   per-name level, recovery correlation rises 0.047 → **0.951**.
4. **AR(1) time-varying state — tried, measured, rejected, retained.**
   `state_innovation_scale` defaults to `0.0`. See §0b for the evidence.
5. **Exported values corrected twice.** The de-standardisation fix removes a **+1.5 to
   +2.3 pp** upside overstatement on every `T>1` run, and `expm1` is now clipped in **log**
   space (`LOG_UPLIFT_CLIP_*`) so the exported `er_*` / price-target columns cannot blow up.
6. **Drift design pruned 21 → 15 columns** (condition number 1,580 → 23, max VIF 162 → 3.8)
   and `region` dropped from the crossed group effects.
7. **§9b model comparison** (`run_model_comparison`) closes the last `❌` in this module's
   Bayesian-workflow coverage row.
8. **Artifact export** lands in a per-section subdirectory tree (0.9.9.13); this notebook
   wires `enable_artifact_export()` + `set_export_section()` behind an opt-in flag.

## §0 — Environment, imports & run configuration

`PYTENSOR_FLAGS` must be set **before** PyTensor/PyMC are first imported (forward slashes
in any `cxx` path — the flag parser is posix shlex and strips backslashes). Importing
`probabilistic_ml_model` normalises the flags via `force_python_vm()`; `set_env.ps1`
handles the full setup. The C backend is **off** project-wide unless
`PML_ENABLE_PYTENSOR_C=1` is set before import.

All workflow knobs live on **`KalmanRunConfig`** (frozen dataclass).
`KalmanRunConfig.from_env()` resolves only five variables — `RANDOM_SEED`,
`KALMAN_PT_RESULTS_DIR`, `KALMAN_PT_EXPORT_DRAWS`, `PML_FIG_WIDTH_PX`, `LOG_LEVEL`.
Everything else (sampling budget, panel geometry, Monte-Carlo screen, CVaR book,
universe-query dates) keeps its dataclass default and is overridden with
`dataclasses.replace` — never by mutation.

In [ ]:
import logging
from dataclasses import replace

import pymc as pm
from sqlalchemy import create_engine

# Section functions live in the module (single source of truth) — import, don't re-define.
import pymc_kalman_filter_pt as kf
from pymc_kalman_filter_pt import (
    # --- config / plumbing ---
    KalmanRunConfig, get_run_config, set_run_config,
    setup_plotting, resolve_db_url,
    enable_artifact_export, set_export_section, get_export_state,
    # --- §1 data + roles ---
    load_kalman_df, load_feature_catalogue, resolve_feature_roles,
    # --- §2/§3/§4 EDA -> features -> panel ---
    run_eda, map_state_space_features, prepare_kalman_panel_inputs,
    KALMAN_PANEL_RESPONSE_EXTRA,
    # --- §5b -> §9b model / inference / diagnostics ---
    build_panel_model, run_prior_predictive, sample_posterior,
    run_posterior_predictive, run_diagnostics, run_model_comparison,
    present_group_effects,
    # --- §10 -> §10c decision layer ---
    KALMAN_SCREEN_LATENT, resolve_screen_latent,
    UPLIFT_CLIP_LO, UPLIFT_CLIP_HI,
    summarize_panel_screen, compute_cvar_aware_book, export_analytics,
    # --- §10K -> §13 side fits ---
    run_universe_kalman_fit,
    run_single_isin_filter, run_single_isin_stochastic_vol,
    run_mingled_cohort_filter, run_mingled_cohort_stochastic_vol,
    run_granular_forest, run_granular_further_views,
    # --- §14 / §14.1 summary + screen visuals ---
    run_summary, run_recommendations,
    plot_screen_overview, plot_risk_return_scatter, plot_top_candidate_forest,
)

cfg = KalmanRunConfig.from_env()
logging.basicConfig(level=cfg.log_level)
setup_plotting()
engine = create_engine(resolve_db_url())
print('Setup complete — orchestrating:', kf.__file__)
cfg

### §0b — Run toggles

**The genuine `(isin, time)` T=4 panel is the DEFAULT.** `panel_lookbacks=('6m','3m','1m')`
builds a real log-uplift history from the `price_target_{lb}_ago` / `price_{lb}_ago` trails,
with the current snapshot as the final step. Collapse to the T=1 cross-section with
`replace(cfg, panel_lookbacks=())` — faster, no time axis, and the per-ISIN intercept is
not created (it is non-identified there).

| knob | default | what it does / why |
|---|---|---|
| `panel_lookbacks` | `('6m','3m','1m')` | T=4 genuine panel. `()` → T=1 snapshot. |
| `draws` | `2000` | Pruning the collinear drift families lifted min ESS **2.1×** at `draws=1000` but landed near 300 against the 400 gate. Nothing collinear remains to cut (drift cond 23, coords orthogonal), so the residual is sampling budget. ≈32 min per T=4 run. |
| `state_innovation_scale` | `0.0` (**off**) | AR(1) time-varying state on top of the intercept. Tried and **rejected** at T=4: +0.013 recovery correlation (0.964 vs 0.951) for min ESS **14 vs 69** and max R-hat 1.13 vs 1.03, with `sigma_state`/`rho` drifting between draw budgets (0.265→0.216, 0.63→0.49) — the "two fits disagree" signature of a non-identified variance component. Set `0.1` to enable; revisit on a longer panel. |
| `isin_level_scale` (builder arg) | `0.10` | Prior scale of `sigma_isin_level`, the per-ISIN random intercept. **This is the layer the panel actually buys.** `0.0` pins it off (the pre-0.9.9.14 baseline). |
| `enable_model_comparison` | `False` | §9b. Refits both arms **and** computes a pointwise `log_likelihood` per arm (~820 MB each at full panel size) → ≈3× sampling cost. |
| `comparison_max_isins` | `800` | ISIN subsample for §9b; the retained fraction is printed so a truncated comparison never reads as a full one. |
| `panel_response_extra` | `()` | Keys of `KALMAN_PANEL_RESPONSE_EXTRA` promoting a **second** response series (`D > 1`) — which is what activates the otherwise-dormant rank-1 ICM. The one supplied series, `pt_dispersion`, drops the collinear drift predictor `feat_pt_noise_drift`. **Off by default:** the `D > 1` path produced the historic R-hat 4.45 / min-ESS 4.3 freeze. |

Three notebook-local flags sit alongside them:

- **`EXPORT_ARTIFACTS`** — `enable_artifact_export()` once, then `set_export_section(...)`
  at the top of every code cell below (CLAUDE.md's notebook convention; there is no
  enclosing `with` block per cell). Off by default, so an interactive run writes nothing
  to `KALMAN_PT_RESULTS_DIR`. `set_export_section` is a harmless no-op while disabled.
- **`WRITE_ANALYTICS`** — §10c's DB write. **Off by default**: it is a DROP-and-RECREATE of
  `analytics.kalman_filtered_price_targets`, the GEIB dashboard's only source. Use
  `scripts/export_kalman_analytics.py` for a production refresh (see the closing cell).
- **`ROBUST`** — Student-t vs Normal panel likelihood; `True` here, see §5.

In [ ]:
# --- notebook-local toggles ---------------------------------------------------
EXPORT_ARTIFACTS = False   # True -> persist figures/tables under KALMAN_PT_RESULTS_DIR
WRITE_ANALYTICS = False    # True -> §10c DROPs and RECREATEs the analytics table
ROBUST = True              # Student-t panel likelihood (matches validation + export)

# --- config overrides (uncomment as needed) ----------------------------------
# cfg = replace(cfg, panel_lookbacks=())                    # collapsed T=1 cross-section
# cfg = replace(cfg, state_innovation_scale=0.1)            # enable the opt-in AR(1) state
# cfg = replace(cfg, panel_response_extra=('pt_dispersion',))  # D=2, activates the ICM
# cfg = replace(cfg, enable_model_comparison=True)          # §9b (~3x sampling cost)
# cfg = replace(cfg, draws=500, tune=500, chains=2)         # smoke run

set_run_config(cfg)

if EXPORT_ARTIFACTS:
    enable_artifact_export()
    print('Artifact export ->', get_export_state().root)
else:
    print('Artifact export OFF (display only).')

_T = len(cfg.panel_lookbacks) + 1 if cfg.panel_lookbacks else 1
print(f'panel_lookbacks : {cfg.panel_lookbacks or "() — collapsed T=1 cross-section"}  (T={_T})')
print(f'state layer     : state_innovation_scale={cfg.state_innovation_scale} '
      f'({"AR(1) ON" if cfg.state_innovation_scale > 0 else "per-ISIN intercept only"})')
print(f'response_extra  : {cfg.panel_response_extra or "() — D=1, ICM dormant"}  '
      f'(supported: {tuple(KALMAN_PANEL_RESPONSE_EXTRA)})')
print(f'NUTS budget     : draws={cfg.draws} tune={cfg.tune} chains={cfg.chains} '
      f'cores={cfg.cores} target_accept={cfg.target_accept}')
print(f'likelihood      : robust={ROBUST} ({"Student-t" if ROBUST else "Normal"})   '
      f'write_analytics={WRITE_ANALYTICS}')

## §1 — Data load & feature-role resolution

- `load_kalman_df(engine, cfg)` — cross-sectional `pml.mv_pymc_kalman_pt` snapshot (one row
  per ISIN). The universe window (`next_earnings >= cfg.min_next_earnings`,
  `income_statement_report_date >= cfg.min_report_date`) is **config-driven** — roll the
  dates forward on `KalmanRunConfig`, never in the query.
- `load_feature_catalogue` — the `kalman_pt` rows of `pml.vw_pymc_feature_catalogue` (the
  SQL registry SSOT). Several `kalman_pt` roles are flipped via per-model overrides in
  `pml.pml_df_feature_alias`, so read the catalogue rather than assuming the base-row role.
- `resolve_feature_roles` — groups columns by `pymc_role` (catalogue SSOT, MV-schema fallback).

> **Not point-in-time.** `mv_pymc_kalman_pt`'s seven `days_*` horizons are computed against
> `CURRENT_DATE`, so a refresh on a different day silently shifts every one. Fine for the
> live screen; unusable as-is for a backtest. It is also part of why the whole `days_*`
> family is barred from the drift matrix (`KALMAN_TIME_COVARIATE_PREFIX`) and feeds the
> `t_scaled` axis instead.

In [ ]:
set_export_section('01_data')

kalman_df = load_kalman_df(engine, cfg)
feature_catalogue = load_feature_catalogue(engine)
roles = resolve_feature_roles(kalman_df, feature_catalogue)
print(f'{kalman_df.shape[0]:,} ISINs x {kalman_df.shape[1]} columns')
kalman_df.head()

## §2 — Exploratory data analysis

`run_eda` renders the EDA panels through a state-space lens: drift features → the
state-transition mean (`beta` slopes), noise wideners → the measurement-noise scale.
The panels carry decision context by default:

- the **industry ridge** is sorted by median with a 0% reference line;
- the **driver facets** annotate each facet with its Spearman ρ (plus an OLS trend line
  when `statsmodels` is available), so signal strength reads off directly;
- the per-coord group forests are **consolidated into one faceted panel**, gated to the
  coords the fused model actually uses as group effects, levels sorted by median with
  universe-median and 0% reference lines.

Note the EDA still shows the *excluded* families (`feat_pt_{median,high,low}_drift`, the
analyst composition legs) — they remain valid EDA / export columns and stay in the MV and
the catalogue. Only the **drift design matrix** drops them; see §3.

In [ ]:
set_export_section('02_eda')

run_eda(kalman_df, roles)

## §3 — State-space feature mapping

`map_state_space_features` maps the catalogue's `kalman_pt` mutable_predictors onto Kalman
roles via `KalmanFilterPriceTarget.select_drift_features`, which applies the SSOT partition
in `KALMAN_DRIFT_EXCLUDED_FEATURES` (`KalmanFilterModel.py`): leakage, noise wideners, named
tilt drivers, support counters, rating counts, the collinear composition leg, Piotroski
components and `days_*` time covariates all stay out of the drift matrix.

### The drift matrix is now 15 columns, not 21

`beta` was the **last failing convergence gate** on the full 6,540-name T=4 validation:
R-hat 1.026 / bulk-ESS 140 against 1.01 / 400, at **zero divergences**. Zero divergences
with slow mixing is the signature of poor conditioning, not bad geometry — and the design
carried condition number **1,580** with a smallest eigenvalue of 0.004. Two families each
restated one signal:

- **`KALMAN_PT_DRIFT_SIBLING_FEATURES`** — `feat_pt_{median,high,low}_drift` are the same
  `pml.target_drift()` run over the median / high / low target trails as `feat_pt_drift`
  runs over the mean. A consensus revision moves the whole band together: r = 0.81–0.89,
  VIF = 162 / 77 / 25 / 6.6.
- **`KALMAN_COLLINEAR_COMPOSITION_FEATURES`** — `feat_analyst_{bullish,bearish,neutral}_pct`
  and `feat_analyst_conviction` are all functions of the same six `num_*_ratings` buckets;
  `conviction` is literally `|bullish − bearish|` and the three pct legs sum to ~1.

One representative survives per family — **`feat_pt_drift`** and **`feat_analyst_rating`**:

| set | k | cond | max VIF | R² |
|---|---|---|---|---|
| before | 21 | 1,580 | 162.5 | 0.6538 |
| after | **15** | **23** | **3.8** | **0.6499** |

69× better conditioning for a 0.6 % relative loss in explanatory power. An orthogonal
replacement for the dropped siblings (`high_drift − low_drift`, band-widening dynamics) was
tried and rejected: it correlates −0.003 with the response and moved R² by 0.0001.

> **v3's note here is retired.** v3 argued that `feat_analyst_conviction` was "clearly
> identified on the genuine T=4 panel (−0.024, 89% ETI excluding 0)" and should stay. It
> stayed null for the right reason: it never carried information the rest of the analyst
> family did not already have. `feat_one_day_return` — the *other* null beta in that note —
> is **kept**: its VIF is ~1, so it is merely uninformative rather than collinear.

> **The exclusion lives in Python, not SQL.** Flipping `pymc_role` to `'excluded'` in
> `pml_df_feature_alias` would drop the row from `vw_pymc_feature_catalogue` while
> `mv_pymc_kalman_pt` still emits the column — making
> `pml.assert_pymc_catalogue_coverage()` raise `MISSING_FROM_CATALOGUE`. The columns also
> remain valid for EDA / the analytics export, and the analyst family is shared with the
> `price_target` model. `kalman_pt` reports 64 OK / 0 violations after the change.

In [ ]:
set_export_section('03_features')

drift_features, mapping = map_state_space_features(kalman_df, feature_catalogue)
print(f'{len(drift_features)} drift features (expected 15 on the current MV):')
for _f in drift_features:
    print('  -', _f)
mapping

## §4 — Fused-panel data containers

`prepare_kalman_panel_inputs` filters to log-space-usable rows and builds the
`KalmanPanelInputs`: the standardised `(isin, time, y_series)` response tensor `Y`, the
standardised time matrix `t_scaled`, the drift design matrix, the noise-widener drivers,
the systematic-risk / size / volume tilt inputs, and the categorical group coords.

**Time axis.** With `panel_lookbacks=('6m','3m','1m')` (the default) each lookback's implied
uplift `price_target_{lb}_ago / price_{lb}_ago − 1` is winsorised and `log1p`-mapped onto the
response scale — a **genuine** oldest→newest history panel (`T = len + 1`) with the snapshot
as the final step. Missing history cells (~1.5 %) are filled with the name's **own** snapshot
uplift, never a cross-sectional-mean fake observation. With `panel_lookbacks=()` the panel is
the collapsed T=1 cross-section and `t_scaled` is the standardised days-to-earnings covariate.

> **Primary response = log uplift.** `feat_log_uplift = log1p(feat_implied_upside)` —
> modelling the log keeps `expected_pt = last_price · exp(log_uplift)` strictly positive.
> `feat_implied_upside` itself is leakage-barred from the drift matrix.

### Two corrections that changed the exported numbers

1. **Support band (`UPLIFT_CLIP_LO` / `UPLIFT_CLIP_HI` = −0.95 / +5.0).** The response is
   winsorised to this decimal band *before* `log1p`, so the model never observes an uplift
   outside it. The two places that map back **out** of log space —
   `panel_posterior_upside`'s `expm1(latent)` and `summarize_panel_screen`'s `expm1(mc)` —
   were previously unbounded, and the 2026-08-10 export shipped `er_mean` up to 7.4e12 and
   `er_sd` up to 1.32e15 for ~1 % of names. Both directions now share one band, clipped in
   **log** space (`LOG_UPLIFT_CLIP_*`), which is sign-preserving and leaves `prob_pos`
   untouched. Read the result honestly: this truncates the posterior to the support the
   model was fit on, it does not extrapolate past it.
2. **Exact de-standardisation.** `KalmanPanelInputs` now carries the fit-time
   `response_mean` / `response_std`, so `_panel_response_stats` inverts the standardisation
   **by construction**. It previously recomputed the moments by tiling the *snapshot*
   column across `T` — correct only for the tile-based panel removed in 0.9.9.10. On the
   6,401-name T=4 run the pooled moments were `(0.207540, 0.249392)` vs the snapshot's
   `(0.224745, 0.241625)`, inflating `expected_upside` by **+2.32 pp** at a −0.5 latent
   through **+1.50 pp** at +1.0. A panel lacking the fields falls back to the legacy
   computation **with a warning**, never silently.

**Optional second response series.** `response_extra` promotes a key of
`KALMAN_PANEL_RESPONSE_EXTRA` to a second series (`D > 1`), activating the rank-1 ICM
(`mu_isin_loading` / `sigma_series`). The supplied series `pt_dispersion` =
`log1p(price_target_stddev_{lb} / price_{lb})` is a distinct signal (disagreement, not
direction) with a genuine `*_ago` trail. Promoting it **drops** the conflicting drift
predictor `feat_pt_noise_drift` (response ↔ predictor disjointness), with a printed note.

In [ ]:
set_export_section('04_panel')

panel = prepare_kalman_panel_inputs(
    kalman_df, roles, drift_features,
    history_lookbacks=cfg.panel_lookbacks,
    response_extra=cfg.panel_response_extra,
)
print('Y shape (isin, time, y_series):', panel.Y.shape)
print('response_names :', panel.response_names)
print('drift_names    :', panel.drift_names)
# Fit-time moments: these make the §10 de-standardisation exact by construction.
print('response_mean  :', getattr(panel, 'response_mean', None))
print('response_std   :', getattr(panel, 'response_std', None))
print(f'uplift support band: [{UPLIFT_CLIP_LO:+.0%}, {UPLIFT_CLIP_HI:+.0%}] (decimal), '
      'clipped in log space on the way back out')

## §5 — Build the fused panel model

`build_panel_model` wraps `build_fused_kalman_pt_model` and renders the model graph.

### Generative form

Per ISIN $i$, time step $t$, response series $d$ (primary $d{=}0$ is `feat_log_uplift`):

**Model A — risk-conditioned drift baseline.** A hierarchical regression on the standardised
drift design with crossed, fixed-scale sum-to-zero group intercepts:

$$\eta_i = X^{\text{drift}}_i\,\beta + \sum_g \big(e^{(g)}\big)_{[i]}
+ \underbrace{\sigma^{\text{lvl}}\, z^{\text{lvl}}_i}_{\text{only when } T>1},\qquad
\beta \sim \mathcal{N}(0,1),\ \ e^{(g)} \sim \text{ZSN}(\sigma{=}0.25)$$

$$\text{risk\_adj\_return}_i = \eta_i - \lambda\,z(\bar\beta_i)
- \gamma\,z(\text{mcap ratio}_i) - \kappa\,z(\text{rel\_volume}_i),\qquad
\text{achieve\_prob}_i = \sigma(\text{state\_now}_i)$$

**Model B — panel spine.**

$$\mu^{\text{reg}}_{i,t,d} = \alpha_{t,d} + W_d\,\text{state\_path}_{i,t},\qquad
y_{i,t,d} \sim \text{StudentT}\big(\nu,\ \mu^{\text{reg}}_{i,t,d},\
\sigma^{\text{isin}}_i \tau_d\big)$$

- $\alpha_{t,d}$ are **T direct per-time intercepts** — exactly identified. The former
  zero-anchored GRW deviations (+ a global slope) were mutually aliased per time slice on an
  isin-constant lookback axis and reproduced the historic scale×innovation ridge (2026-08-01
  T=4 run: 190 divergences, `alpha_level` R-hat 1.06 — 0 divergences after the fix).
- The per-series time slope $\beta^{(t)}_d$ is **not materialised at all** on an
  isin-constant axis. It was previously published as a Deterministic pinned at 0, which read
  as a fitted-and-dead parameter, gave arviz a constant to divide 0/0 on, and drew a flat
  zero line in the §13b slope panel. It returns unchanged when `t_scaled` genuinely varies.
- $W_d$ is the sign-fixed coregion loading (primary ≡ 1); $\tau_d$ the per-series noise
  diagonal; $\nu \ge 2.5$ the tail dof. Both are inert while `D == 1`.
- $\text{state\_path}_{i,t} = \mu^{\text{isin}}_i$ with the AR layer off, and
  $\text{state\_now}_i = \text{state\_path}_{i,-1}$.

**Crossed group effects** are `_FUSED_KALMAN_GROUP_EFFECTS = ('trading_region', 'sector',
'style_class', 'size_class')` at a **fixed** `GROUP_EFFECT_SCALE = 0.25` — the group SD is
not learned (structurally non-identified on one slice). `region` was **dropped**: it and
`trading_region` agree for 96.12 % of the universe (Cramér's V 0.938, only cross-listings
differ) against V ≤ 0.24 for every other pair, and once the per-ISIN intercept landed they
became the two worst-mixing globals. `trading_region` is kept because listing venue
determines analyst coverage and currency — the mechanism the drift features measure.

> **The `robust` trap.** `robust=True` (Student-t) is `build_fused_kalman_pt_model`'s own
> default, and it is what `scripts/validate_kalman_state.py` and
> `scripts/export_kalman_analytics.py` fit. But `pymc_kalman_filter_pt.main()` defaults to
> **`robust=False`** (Normal). Exporting on the Normal variant ships numbers no gate has
> seen, so this notebook sets `ROBUST = True` in §0b — pass `robust=True` explicitly if you
> ever call `main()` directly.

In [ ]:
set_export_section('04_panel')

VOLUME_PENALTY = 0.25  # HalfNormal prior scale for the learned volume_loading tilt; 0.0 disables
model = build_panel_model(panel, robust=ROBUST, volume_penalty=VOLUME_PENALTY, config=cfg)
print('state layer vars:',
      [v for v in ('state_path', 'state_now', 'sigma_isin_level', 'sigma_state')
       if v in model.named_vars])
pm.model_to_graphviz(model)

## §6 — Prior predictive checks

`run_prior_predictive` draws `cfg.prior_draws` prior samples of `expected_return`, the
decision latent (resolved through `resolve_screen_latent`, i.e. `state_now`), `achieve_prob`
and `sigma_isin`, then de-standardises onto the interpretable **percent** implied-upside
scale and compares against the empirical distribution — the Bayesian-workflow stage
contract, not a density doodle.

`state_path` is deliberately **skipped**: an `(isin, time)` tensor over prior draws is large
and carries no extra prior information beyond `state_now`.

In [ ]:
set_export_section('06_prior')

prior_idata = run_prior_predictive(model, panel, cfg)
print('decision latent:', KALMAN_SCREEN_LATENT,
      '->', resolve_screen_latent(prior_idata.prior).name)
prior_idata.prior

## §7 — Posterior inference (NUTS)

`sample_posterior` tries `nutpie → numpyro → pymc` in priority order (the same order
`sample_with_fallback` uses for the §9b arms and the validation script) and merges the prior
groups into the posterior `DataTree`. The budget comes from the config:
`draws/tune/chains/target_accept/random_seed`. Passing `panel=` stamps the drift-feature
aliases plus their catalogue metadata onto `constant_data['drift_features']` via
`stamp_feature_provenance`.

> **`cores=1` in the IDE kernel.** Launching nutpie's parallel native workers inside an
> IDE-managed Jupyter kernel on Windows can crash the kernel process — an uncatchable native
> crash that surfaces only as *"Connection to IDE-Managed Server is lost"*. Chains run
> sequentially here; the standalone script path uses `cfg.cores`.

> **Sampler choice is not a micro-optimisation.** The project forces the PyTensor
> pure-Python VM, under which PyMC's own NUTS produced **zero draws in 42 minutes of CPU**
> on the local-level panel (~16.8k extra parameters on a 5.6k-ISIN T=4 panel). Anything
> sampling this model must go through `sample_posterior` / `sample_with_fallback`, never
> bare `build_sample_kwargs` (whose `nuts_sampler=None` default lands on that path).

**Validation history** (why the settings look like this):

| run | geometry | result |
|---|---|---|
| T=1 baseline | direct intercepts, no time axis | 0 div, R-hat ≤ 1.01, ESS > 400 |
| T=4 (2026-07-31/08-01, GRW deviations) | aliased level/slope/innovation block | 315 / 190 divergences, `alpha_level` R-hat 1.06 |
| T=4 reparameterised (2026-08-01) | per-time direct intercepts | 0 div, worst R-hat 1.00, ESS ≈ 1.6k, 15.7 min |
| T=4 + per-ISIN latent, 21 drift cols | `draws=1000` | 0 div, but `beta` R-hat 1.026 / ESS 140 |
| T=4, 15 drift cols, `region` dropped | `draws=1000` | 0 div, `beta` R-hat ≤ 1.0121 / ESS 236–296 |
| **T=4, 15 cols, `draws=2000`** | **shipped default** | **0 div, max R-hat 1.0090, min ESS 678.3, ≈32 min** |

`log_likelihood` is hard-coded **off** here (it roughly doubles idata size and no production
path consumes it), so `az.loo` / `az.compare` raise on this object. Attach it post-hoc with
`attach_log_likelihood(idata, model)` — the `idata_kwargs={'log_likelihood': True}` route is
silently stripped under nutpie. §9b does exactly that.

In [ ]:
set_export_section('07_posterior')

idata = sample_posterior(model, prior_idata, cores=1, panel=panel, config=cfg)
print('Group effects fitted:', present_group_effects(idata))
print('Divergences:', int(idata.sample_stats['diverging'].sum()))
idata.posterior

## §8 — Posterior predictive checks

Calibration of the standardised `(isin, time, y_series)` likelihood:

- **Pooled ECDF overlay** — replicate ECDFs vs the observed ECDF.
- **t-stat calibration** (`mean`, `std`) — observed T(y) inside the replicated distribution.
- **94 % coverage per `y_series`** — printed and charted against the 0.94 target line.
- **94 % coverage per time step** — *new in 0.9.9.14*, and the statistic that tests the
  state layer specifically. Pooled coverage can look correct while the model is
  over-confident at the oldest lookbacks and over-dispersed at the snapshot; a **monotone
  drift across `t`** indicates a mis-set innovation scale. This is what falsified the
  literal cumulative random walk: its marginal variance grows as √t and the full-scale run
  produced 89.9 % → 95.3 % → 97.2 % → 98.2 % against a 94 % target. The shipped build
  measured 91.6–92.5 % across `t` — flat, no drift.
- **PIT ECDF** — uniform / in-band when calibrated.

In [ ]:
set_export_section('08_ppc')

run_posterior_predictive(model, idata, panel)

## §9 — MCMC diagnostics

`run_diagnostics` reports R-hat / bulk-&-tail ESS, divergences, trace / rank-dist / forest
views, the NUTS energy, prior→posterior contraction, the ESS evolution and the variance
partition (printed **and** rendered as a stacked share bar). Gates follow Vehtari et al.
(2021): **R-hat < 1.01**, **ESS > `MIN_ESS_GATE` = 400**.

The variable groupings are the module's SSOT:

- **`FUSED_SCALAR_VARS`** — `sigma_base`, `nu`, `sigma_state`, plus the learned sign-fixed
  `risk_loading` / `size_loading` / `volume_loading`. Absent vars are skipped, so
  `sigma_state` appears only when the AR(1) layer is enabled on a `T > 1` panel. **Watch it:
  a posterior pressed against 0 means the panel carries no per-name time dynamics and the
  state layer is dead weight.**
- **`FUSED_VECTOR_VARS`** — `beta`, `alpha_level`, `beta_slope`, `mu_isin_loading`,
  `sigma_series`. `beta_slope` exists only when `t_scaled` varies across ISINs;
  `mu_isin_loading` / `sigma_series` only when `D > 1`.
- Per-coord `sigma_<coord>` are appended at runtime from the coords actually present.

`sigma_alpha_innov` / `sigma_beta_innov` no longer exist (the per-time direct intercepts
removed them). Constant posterior variables are filtered through `_degenerate_posterior_vars`
**before** the `azs.rhat` / `azs.ess` sweep, so the `invalid value encountered in scalar
divide` warning is gone at source rather than suppressed.

Separately, watch **`sigma_isin_level`**: it is the per-ISIN random intercept the T>1 panel
buys, and a collapse toward 0 means the panel carries no per-name signal beyond the drift
regression. `scripts/validate_kalman_state.py` gates on exactly that.

In [ ]:
set_export_section('09_diagnostics')

run_diagnostics(idata, panel)

## §9b — Model comparison (ELPD / LOO) — opt-in

`run_model_comparison` closes the last `❌` in this module's Bayesian-workflow coverage row.
The two arms differ in exactly one thing — whether the per-ISIN latent may evolve over the
panel:

- **`local_level`** — `state_innovation_scale` from the config (use `0.1`);
- **`static`** — `state_innovation_scale=0.0`, pinning the state at its t=0 anchor, i.e. the
  pre-0.9.9.14 time-constant build.

Both are refit on the same subsampled panel so the ELPD contrast is like-for-like.

**Why it is opt-in:** each arm needs a pointwise `log_likelihood` group of
`chains × draws × n_isin × T × D` floats — **~820 MB per arm** at full panel size — and both
arms are refit, so this roughly **triples** the run's sampling cost. `comparison_max_isins`
(default 800) bounds the ISIN axis and the retained fraction is printed, so a truncated
comparison never reads as a full one. Needs `T > 1`; it prints a skip on a T=1 panel.

**Reading it:**

- The group is attached post-hoc with `attach_log_likelihood` (`pm.compute_log_likelihood`).
  The `idata_kwargs={'log_likelihood': True}` route does **not** work — nutpie ignores
  `idata_kwargs` and `build_sample_kwargs` strips it.
- ArviZ 1.x exposes the value as **`.elpd`**. `.elpd_loo` was removed and a `getattr`
  fallback on the old name silently yields `nan`, even though the `ELPDData` repr still
  prints the `elpd_loo` row label.
- Judge on **`elpd_diff` vs `dse`**, not on Pareto k-hat. High k-hat on the state arm is
  expected by construction (it carries a per-ISIN latent path) — the documented weakness of
  PSIS-LOO for models with per-observation latents. A margin inside ~2 `dse` is
  inconclusive, not a win.

In [ ]:
set_export_section('09b_comparison')

# OPT-IN: refits BOTH arms and computes a pointwise log_likelihood for each
# (~820 MB per arm at full panel size) -> roughly 3x this run's sampling cost.
# Uncomment to run.
#
# cmp_df = run_model_comparison(
#     panel,
#     config=replace(cfg, enable_model_comparison=True, state_innovation_scale=0.1),
#     robust=ROBUST, volume_penalty=VOLUME_PENALTY,
# )
# cmp_df

print('§9b skipped (opt-in). Uncomment the call above, or set '
      'cfg = replace(cfg, enable_model_comparison=True) before kf.main(...).')

## §10 — Expected price targets: posterior screen

`summarize_panel_screen` → a `ScreenContext` with the posterior `expected_upside` /
`expected_pt` draws, the per-ISIN screening `results` table and the structural-TS
Monte-Carlo summary. The `er_*` columns are genuine **decimal returns** (0.25 = +25 %);
percent scaling happens only at display boundaries. The Monte-Carlo horizon / damping come
from `cfg.mc_horizon` / `cfg.mc_rho`.

> **The decision latent is `state_now`.** Since the state layer landed, every consumer
> resolves the per-ISIN quantity through `resolve_screen_latent` — the **filtered level at
> the final (snapshot) time step**. The posterior *variable* `risk_adj_return` is now only
> the t=0 structural anchor. The screen and export **column** named `risk_adj_return` keeps
> its name and units but reports the filtered level; the two coincide exactly when `T == 1`
> or the state is pinned off, so the fallback is not a degraded path.

> **De-standardisation is exact, and `expm1` is bounded.** The inverse uses the panel's
> fit-time `response_mean` / `response_std` (§4), removing the +1.5–2.3 pp overstatement,
> and the log-space draws are clipped to `[LOG_UPLIFT_CLIP_LO, LOG_UPLIFT_CLIP_HI]` before
> `expm1`. Clipping in log space is sign-preserving, so `prob_pos` is untouched. Names whose
> distribution pins at the cap are flagged in §10c rather than published with a fabricated
> ranking score.

Renders: the per-industry posterior forest (0-line, sorted), the fused-model internals panel,
and the comparative-returns views (percent-space shrinkage scatter, distributional KDE
overlay, per-sector forest).

In [ ]:
set_export_section('10_screen')

screen = summarize_panel_screen(idata, panel, horizon=cfg.mc_horizon, rho=cfg.mc_rho)
results = screen.results
results.head(15)

### §10b — CVaR-aware risk analytics & sizing (RiskBook)

Single source of truth for the risk layer (`RiskBookModel.compute_cvar_aware_book`): per-name
expected shortfall (CVaR of the posterior upside draws), a reward-to-CVaR (STARR) ranking,
and a per-name-capped long book with joint-draw portfolio aggregates. Knobs resolved from the
config: `cvar_alpha` / `weight_cap` / `k_book` / `p_long` / `mcap_country_r_max`. The same
`RiskBook` feeds the §10c export and the §14b recommendations.

All columns are raw decimals, including `cvar05` and `exp_vol`.

In [ ]:
set_export_section('10b_risk')

risk_book = compute_cvar_aware_book(idata, panel, screen, results, config=cfg)
print({k: round(v, 4) if isinstance(v, float) else v
       for k, v in risk_book.summary.items()})
risk_book.book.head(25)

### §10c — Analytics export & decision dashboard

`export_analytics` maps the posterior onto `analytics.kalman_filtered_price_targets`
(raw-decimal convention). The overview dashboard carries the portfolio star (aggregate
E[r] / vol / CVaR from `RiskBook.summary`), the held-name efficient hull on the risk-return
map, the MC return fan over the sized book, and the shrinkage view on signed-log axes.

**New: the `out_of_support` flag.** A name whose forward-return distribution is pinned at the
+500 % uplift cap has no reliable ranking metric — its `er_sd` collapses toward 0, which
makes `expected_sharpe_ratio = er_mean / er_sd` explode. A Sharpe of 717 sorts straight to
the top of any risk-adjusted screen while marking precisely the names the model understands
*least*. So for those rows `expected_sharpe_ratio`, `reward_to_cvar` and `cvar_book_weight`
go NULL (weight re-filled to 0) and `out_of_support = TRUE`; identity, price targets and the
raw `er_*` distribution are retained, so nothing vanishes silently.

The detector tests **`er_p05`, not `er_mean`**. `er_mean` averages the clipped draws, so a
handful landing below the cap drag it ~1e-4 under — a first attempt on `er_mean` matched
**zero** of the 18 affected names. `er_p05` lands exactly on the cap once ~95 % of the draws
are clipped, which is the condition worth calling out of support. Verified: excluding those
18 names takes max Sharpe 717.7 → 10.1 and max reward-to-CVaR 500 → 14.3.

> **0.9.9.14 changes the exported VALUES, not the schema.** Two corrections compound — the
> de-standardisation fix removes a +1.5–2.3 pp upside overstatement on every `T>1` run, and
> the per-ISIN latent widens the `expected_upside` HDIs (the previous build was ~5×
> over-confident). The layout and the raw-decimal convention are unchanged, so **no DDL
> migration is needed** — but the export and the GEIB dashboard deploy still ship as a pair.

`WRITE_ANALYTICS` is `False` by default: the write is a DROP-and-RECREATE of the dashboard's
only source. For a production refresh, run `scripts/validate_kalman_state.py` (exit 0
required), then `scripts/export_kalman_analytics.py` — see the closing cell.

In [ ]:
set_export_section('10c_analytics')

kalman_results = export_analytics(idata, panel, screen, risk_book=risk_book,
                                  write=WRITE_ANALYTICS)
if 'out_of_support' in kalman_results.columns:
    print(f'out_of_support: {int(kalman_results["out_of_support"].sum())} '
          f'of {len(kalman_results)} names (ranking metrics NULLed, er_* retained)')
kalman_results.head()

### §10K — Universe-consensus fit

`run_universe_kalman_fit(kalman_df)` is the one-call driver `main()` uses: it pools every
row's `price_target*_ago` trail into a weekly-median consensus series
(`build_universe_consensus`), routes it through the canonical `fit_kalman_model` — the
funnel-free marginalized GRW (+ trend), nutpie, spot-anchored at the universe-median
`last_price` — structurally forecasts to the universe-median fiscal events, and calls
`report_universe_kalman_fit` internally.

v3 hand-assembled these three steps in the notebook; the driver is the SSOT, so this cell is
one call. Override any fit kwarg by keyword (e.g. `samples=`, `chains=`, `trend=`).

The structural forecast renders in **return space** — observed vs expected returns per
fiscal event with a 0 % break-even line — and the forecast table always carries
`implied_upside_pct`.

In [ ]:
set_export_section('10k_universe')

universe_fit = run_universe_kalman_fit(kalman_df, random_seed=cfg.random_seed)
universe_fit

## §11 — Single-ISIN time-series Kalman filter (+ §11b stochastic volatility)

The literal single-security GRW filter on the richest `*_ago` history (time axis anchored on
`income_statement_report_date`). The candidate pull is config-driven
(`min_mcap_country_rank`, `candidate_limit`). The structural forecast is the **return-space**
panel (`plot_kalman_forecast_returns`): observed implied returns, the smoothed
implied-upside band, nested latent/predictive forecast bands per fiscal event (the gap
between them is the analyst observation noise), per-horizon +X % annotations.

§11b refits with stochastic volatility — its σ_obs(t) path renders as the companion row
(posterior median).

In [ ]:
set_export_section('11_single_isin')
single_ctx = run_single_isin_filter(panel.frame, engine, cfg)

set_export_section('11b_single_sv')
run_single_isin_stochastic_vol(single_ctx)

## §12 — Mingled-ISIN earnings-window cohort filter (+ §12b stochastic volatility)

Every ISIN whose `next_earnings` lands within ±`cfg.earnings_window_days` of today is
unpivoted and the cross-sectional **median** target taken per shared as-of date — one
earnings-cohort consensus series, fit with the marginalized GRW (+ trend). Return-space
forecast + upside-bearing forecast table, as in §11.

In [ ]:
set_export_section('12_mingled')
mingled_ctx = run_mingled_cohort_filter(panel.frame, engine, cfg)

set_export_section('12b_mingled_sv')
run_mingled_cohort_stochastic_vol(panel.frame, mingled_ctx)

## §13 — Granular earnings-cohort posterior-predictive forest (+ §13.1 further views)

Keeps the §12 cohort definition but stays per-ISIN granular, reusing the fitted fused
posterior (no refit): per-name `expected_pt` posterior forests with the raw analyst targets
overlaid and pooled reference bands.

> **§13b panel (d) changed.** It now plots the **`state_path` median with a 10–90 %
> cross-sectional band** instead of the `beta_t` slope, falling back to `beta_t` only on a
> genuinely isin-varying time axis. Read it directly: a visibly **widening** band from t=0 is
> the state layer working; a **flat** one means `sigma_state` collapsed and the AR layer is
> dead weight. With the AR layer off (the default) the path is constant across `t` by
> construction — `state_path` equals `mu_isin` — so a flat band there is expected, not a
> failure.

In [ ]:
set_export_section('13_forest')
forest_ctx = run_granular_forest(idata, results, panel, screen, engine, cfg)

set_export_section('13b_further_views')
run_granular_further_views(prior_idata, panel, screen, forest_ctx)

## §14 — Comprehensive summary & actionable recommendations

`run_summary` consolidates the run into an earnings-cohort vs baseline vs universe read; the
cross-sectional table and the sector mix render as decision panels (grouped metric bars;
cohort-vs-universe sector-tilt diverging bar).

`run_recommendations` (§14b) turns the posterior into risk-aware signals. The
group-allocation block renders the **shrunk-excess forest** (per-coord OW/UW bands,
verdict-coloured); the CVaR sizing block renders the **book composition** chart with the
portfolio aggregates (E[upside] / CVaR5 / reward-to-CVaR / diversification).

In [ ]:
set_export_section('14_summary')
run_summary(results, screen, forest_ctx, mingled_ctx)

set_export_section('14b_recommendations')
run_recommendations(idata, panel, results, screen, forest_ctx, risk_book=risk_book)

### §14.1 — Screen overview, risk/return screen & top-candidate forest

These plotting functions live in the module — import, don't re-define:

- `plot_screen_overview` — upside distribution + top-N ranked names with HDIs;
- `plot_risk_return_scatter` — interactive upside vs posterior-uncertainty screen
  (colour = sector, size = market cap);
- `plot_top_candidate_forest` — posterior expected-upside forest of the top names.

In [ ]:
set_export_section('14_summary')

plot_screen_overview(results, top_n=50)
plot_risk_return_scatter(results)
plot_top_candidate_forest(screen, results, top_n=50)

---
### Artifacts available at the top level

`cfg`, `kalman_df`, `roles`, `drift_features`, `panel`, `model`, `prior_idata`, `idata`,
`screen`, `results`, `risk_book`, `kalman_results`, `universe_fit`, `single_ctx`,
`mingled_ctx`, `forest_ctx`.

### One-shot alternatives

```python
# Full workflow in one call. NOTE main() defaults to robust=False (Normal likelihood)
# while the builder, the validation script and the export script all use Student-t.
kf.main(run_eda_section=True,write_analytics=False,robust=True,export_results=True,config=cfg)
# -> {'idata', 'prior_idata', 'results', 'kalman_results',
#     'panel', 'screen', 'risk_book', 'universe_fit'}
```

```powershell
. .\set_env.ps1

# Gate the re-export: convergence, sigma_isin_level alive, the predicted
# sigma_base-falls / per-name-sd-widens signature, per-time PPC coverage,
# and the de-standardisation delta. Must exit 0.
python scripts\validate_kalman_state.py

# Production refresh of analytics.kalman_filtered_price_targets (robust=True,
# stops after §10c). Deploy the GEIB dashboard after it completes — they ship as a pair.
python scripts\export_kalman_analytics.py --dry-run
python scripts\export_kalman_analytics.py

# One-off migration of a pre-0.9.9.13 flat results directory into the section tree.
python pymc_kalman_filter_pt.py --migrate-layout
python pymc_kalman_filter_pt.py --migrate-layout --apply
```

Artifacts (PNG / CSV / SQL / JSON / NetCDF) land under `KALMAN_PT_RESULTS_DIR` in the
per-section subdirectories resolved by `_export_dir_for` against the `_EXPORT_SECTION_DIRS`
SSOT — never build a result path by hand. The curated bulk frames in `_SQL_EXPORT_ARTIFACTS`
additionally become `analytics."<stem>"` tables plus a generated `<stem>.sql` DDL file;
`KALMAN_PT_SQL_EXPORT=0` (or an unreachable database) falls back to CSV while still emitting
the DDL.